# Sagokompis – röst till Frans och Nallen
Gör ljudfiler (en per mening) med samma röst som provet. Gratis i Google Colab.

**Så gör du:**
1. Meny **Körtid → Ändra körtidstyp → T4 GPU → Spara.**
2. Tryck **Körtid → Kör alla.**
3. När den frågar: välj de två filerna (`texter-frans-nalle.json` och `referensrost.wav`) och tillåt Google Drive.
4. Låt fliken vara öppen. Ljudet sparas i Drive-mappen **sagokompis-rost**. Tar gratistiden slut: kör bara om, den fortsätter där den slutade.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'Ingen GPU: gå till Körtid > Ändra körtidstyp > T4 GPU'
!pip -q install omnivoice

In [ ]:
from google.colab import files, drive
up = files.upload()   # välj texter-frans-nalle.json och referensrost.wav
drive.mount('/content/drive')
import os
UT = '/content/drive/MyDrive/sagokompis-rost'
os.makedirs(UT, exist_ok=True)

In [ ]:
import json, time, torch, torchaudio, subprocess
from omnivoice.models.omnivoice import OmniVoice

REF_TEXT = 'Det var en gång en liten drake som bodde högt uppe på ett berg. Varje kväll tittade månen ner och viskade godnatt.'
texter = json.load(open('texter-frans-nalle.json', encoding='utf-8'))
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda', dtype=torch.float16)
prompt = model.create_voice_clone_prompt(ref_audio='referensrost.wav', ref_text=REF_TEXT)
print(len(texter), 'meningar att göra')

In [ ]:
def gor(post):
    mp3 = f"{UT}/{post['id']}.mp3"
    if os.path.exists(mp3): return False
    torch.manual_seed(1234)
    a = torch.as_tensor(model.generate(text=post['text'], language='sv', voice_clone_prompt=prompt)[0]).float().cpu()
    a = a.reshape(1, -1) if a.dim() == 1 else a
    torchaudio.save('/content/tmp.wav', a, model.sampling_rate)
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i','/content/tmp.wav','-ac','1','-b:a','64k',mp3], check=True)
    return True

# Provkörning: 5 meningar, så ser du hur snabbt det går.
t0 = time.time(); n = 0
for p in texter[:5]:
    n += gor(p)
print('5 meningar på', round(time.time()-t0), 'sekunder. Lyssna på filerna i Drive-mappen innan du kör resten.')

In [ ]:
# Resten. Går att köra om om det avbryts, färdiga hoppas över.
t0 = time.time(); klara = 0
for i, p in enumerate(texter):
    if gor(p): klara += 1
    if i % 25 == 0:
        sek = time.time() - t0
        print(f'{i}/{len(texter)} ({klara} nya) {round(sek/60,1)} min')
print('KLART')